In [1]:
#"import files and required data for training"
import pandas as pd
import os

# Path to SCRIP folder
data_path = "C:/Users/HIMANSHU/Desktop/Ml Project/DATASET/RAW/Multistock/Datasets/SCRIP"

all_files = []

# Collect all CSV files
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".csv"):
            full_path = os.path.join(root, file)
            all_files.append(full_path)

print(f"Total stock files found: {len(all_files)}")

Total stock files found: 1756


In [2]:
#"Combine in a single dataframe"
df_list = []

for file in all_files:
    try:
        df = pd.read_csv(file)
        df_list.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")


data = pd.concat(df_list, ignore_index=True)

print(data.shape)
data.head()

(5237842, 15)


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2008-10-06,20MICRONS,EQ,55.00,80.0,80.0,31.60,33.55,33.65,41.27,11750865,4.849072e+13,NaN,991550.0,0.0844
1,2008-10-07,20MICRONS,EQ,33.65,32.0,38.0,27.85,30.05,30.10,31.51,4556711,1.435988e+13,NaN,333621.0,0.0732
2,2008-10-08,20MICRONS,EQ,30.10,28.0,29.2,25.10,26.40,26.50,26.85,1232192,3.309046e+12,NaN,92240.0,0.0749
3,2008-10-10,20MICRONS,EQ,26.50,24.9,24.9,21.65,23.65,23.20,23.50,603964,1.419308e+12,NaN,113128.0,0.1873
4,2008-10-13,20MICRONS,EQ,23.20,24.3,26.6,23.30,24.70,24.65,25.37,449346,1.140129e+12,NaN,93847.0,0.2089


In [3]:
#"Keep only Required columns"
required_cols = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]

data = data[required_cols]

In [4]:
#"Convert data properly"
data["Date"] = pd.to_datetime(data["Date"])

In [5]:
#"Drop missing values"
data = data.dropna()

In [6]:
#"Sort data"
data = data.sort_values(by=["Symbol", "Date"])

In [7]:
#"keep data only of a specific timeline"
data = data[
    (data["Date"] >= "2000-01-01") &
    (data["Date"] <= "2021-12-31")
]

In [8]:
#"Feature enginnering"
# 1. Base return
data["Return_1"] = data.groupby("Symbol")["Close"].pct_change()

# 2. Momentum
data["Return_5"] = data.groupby("Symbol")["Close"].pct_change(5)
data["Return_20"] = data.groupby("Symbol")["Close"].pct_change(20)

# 3. Moving averages
data["MA20"] = data.groupby("Symbol")["Close"].transform(lambda x: x.rolling(20).mean())
data["MA50"] = data.groupby("Symbol")["Close"].transform(lambda x: x.rolling(50).mean())
data["MA_ratio"] = data["MA20"] / data["MA50"]

# 4. Volatility
data["Volatility"] = data.groupby("Symbol")["Return_1"].transform(lambda x: x.rolling(20).std())

# 5. Volume spike
data["Avg_Volume_10"] = data.groupby("Symbol")["Volume"].transform(lambda x: x.rolling(10).mean())
data["Volume_spike"] = data["Volume"] / data["Avg_Volume_10"]


In [9]:
data = data.dropna()

In [10]:
#"Create target"
data["Target"] = data.groupby("Symbol")["Return_1"].shift(-1)
data["Target"] = (data["Target"] > 0).astype(int)

In [11]:
#"Define features"
features = [
    "Return_5",
    "Return_20",
    "MA_ratio",
    "Volatility",
    "Volume_spike"
]

X = data[features]

In [12]:
#"Define target"
y = data["Target"]

In [13]:
#"Train-test Split"
split_date = "2018-01-01"

X_train = X[data["Date"] < split_date]
X_test = X[data["Date"] >= split_date]

y_train = y[data["Date"] < split_date]
y_test = y[data["Date"] >= split_date]

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Features
features = [
    "Return_5",
    "Return_20",
    "MA_ratio",
    "Volatility",
    "Volume_spike"
]

# Target
data["Target"] = data.groupby("Symbol")["Return_1"].shift(-1)
data["Target"] = (data["Target"] > 0).astype(int)

# Split
split_date = "2018-01-01"

train = data[data["Date"] < split_date]
test = data[data["Date"] >= split_date]

X_train = train[features]
y_train = train["Target"]

# 🔥 SCALER
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 🔥 MODEL
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [19]:
# Create test dataframe again
test = data[data["Date"] >= split_date].copy()

# Predictions
X_test_scaled = scaler.transform(X_test)
test["Prob"] = lr_model.predict_proba(X_test_scaled)[:, 1]

# Ranking
test["Rank"] = test.groupby("Date")["Prob"].rank(ascending=False)

# Signal (top 10 long)
test["Signal"] = 0
test.loc[test["Rank"] <= 10, "Signal"] = 1

# Weights
test["Weight"] = test.groupby("Date")["Signal"].transform(lambda x: x / x.sum())

# Future return
test["Future_Return"] = test.groupby("Symbol")["Return_1"].shift(-1)

# Strategy return
test["Strategy_Return"] = test["Weight"] * test["Future_Return"]

# Daily returns
daily_returns = test.groupby("Date")["Strategy_Return"].sum()
daily_returns = daily_returns.fillna(0)

# ✅ FINAL
cumulative_returns = (1 + daily_returns).cumprod()

In [43]:
nifty = pd.read_csv(
    r"C:\Users\HIMANSHU\Desktop\Ml Project\DATASET\RAW\Multistock\Datasets\INDEX\NIFTY 50.csv"
)

nifty["Date"] = pd.to_datetime(nifty["Date"], dayfirst=True, errors="coerce")
nifty = nifty.dropna(subset=["Date"])

# 🔥 FILTER FIRST
nifty = nifty[nifty["Date"] >= "2018-01-01"].copy()

# Sort
nifty = nifty.sort_values("Date")

# Returns
nifty["Return"] = nifty["Close"].pct_change().fillna(0)

# Cum returns
nifty["nifty"] = (1 + nifty["Return"]).cumprod()

# 🔥 NORMALIZE AFTER FILTER
nifty["nifty"] = nifty["nifty"] / nifty["nifty"].iloc[0]

# Final dataframe
nifty_df = nifty[["Date", "nifty"]]

In [36]:
perf_df = pd.merge_asof(
    strategy_df.sort_values("Date"),
    nifty_df.sort_values("Date"),
    on="Date",
    direction="backward"
)


In [37]:
nifty = nifty[nifty["Date"] >= "2018-01-01"].copy()

nifty = nifty.sort_values("Date")

nifty["Return"] = nifty["Close"].pct_change()

nifty_cum = (1 + nifty["Return"]).cumprod()

# Normalize (important)
nifty_cum = nifty_cum / nifty_cum.iloc[0]

# Set index
nifty_cum.index = nifty["Date"]

print(nifty_cum.head())

Date
2018-01-01   NaN
2018-01-02   NaN
2018-01-03   NaN
2018-01-06   NaN
2018-01-08   NaN
Name: Return, dtype: float64


In [48]:
# Strategy dataframe
strategy_df = pd.DataFrame({
    "strategy": cumulative_returns
})

# Ensure datetime
strategy_df.index = pd.to_datetime(strategy_df.index)

# 🔥 Merge using LEFT JOIN (strategy is master)
perf_df = strategy_df.merge(
    nifty_df,
    left_index=True,
    right_index=True,
    how="left"
)

# 🔥 Fill missing NIFTY values
perf_df["nifty"] = perf_df["nifty"].ffill()
perf_df["nifty"] = perf_df["nifty"] / perf_df["nifty"].iloc[0]

# Reset index
if "Date" in perf_df.columns:
    perf_df = perf_df.rename(columns={"Date": "date"})
else:
    perf_df = perf_df.reset_index().rename(columns={"index": "date"})

# Verify
print(perf_df.head())

                     strategy date  nifty
Date                                     
2018-01-01 00:00:00  1.041215  NaT    NaN
2018-01-02 00:00:00  1.053396  NaT    NaN
2018-01-03 00:00:00  1.051971  NaT    NaN
2018-01-04 00:00:00  1.040435  NaT    NaN
2018-01-05 00:00:00  1.045044  NaT    NaN


In [45]:
# Convert to proper dataframe with Date column
strategy_df = pd.DataFrame({
    "Date": cumulative_returns.index,
    "strategy": cumulative_returns.values
})

strategy_df["Date"] = pd.to_datetime(strategy_df["Date"])
strategy_df = strategy_df.sort_values("Date")

nifty_df["Date"] = pd.to_datetime(nifty_df["Date"])
nifty_df = nifty_df.sort_values("Date")

# 🔥 FINAL FIX
perf_df = pd.merge_asof(
    strategy_df,
    nifty_df,
    on="Date",
    direction="backward"
)

# Rename
perf_df = perf_df.rename(columns={"Date": "date"})

# Verify
print(perf_df.head())

        date  strategy     nifty
0 2018-01-01  1.041215  1.000000
1 2018-01-02  1.053396  1.055709
2 2018-01-03  1.051971  1.002185
3 2018-01-04  1.040435  1.002185
4 2018-01-05  1.045044  1.002185


C:\Users\HIMANSHU\AppData\Local\Temp\ipykernel_10956\2354792140.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nifty_df["Date"] = pd.to_datetime(nifty_df["Date"])


In [41]:
perf_df = pd.DataFrame({
    "date": cumulative_returns.index,
    "strategy": cumulative_returns.values,
    "nifty": nifty_cum.reindex(cumulative_returns.index).values
})

perf_df.to_csv("performance.csv", index=False)

PermissionError: [Errno 13] Permission denied: 'performance.csv'

In [31]:
print("Strategy range:")
print(cumulative_returns.index.min(), "→", cumulative_returns.index.max())

print("\nNIFTY raw range:")
print(nifty["Date"].min(), "→", nifty["Date"].max())

print("\nSample NIFTY dates:")
print(nifty["Date"].head())

Strategy range:
2018-01-01 00:00:00 → 2021-07-01 00:00:00

NIFTY raw range:
2018-01-01 00:00:00 → 2021-12-05 00:00:00

Sample NIFTY dates:
6644   2018-01-01
6666   2018-01-02
6685   2018-01-03
6747   2018-01-06
6790   2018-01-08
Name: Date, dtype: datetime64[ns]


In [52]:
perf_df.to_csv("performance.csv", index=False)